In [1]:
import os
os.chdir("../")

os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/gulmohargautam2005/Deep_Learning.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "gulmohargautam2005"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "49f97a6b0fed094b8759046225cdc350db137a6c"  # from Step 2

In [12]:
%pwd

'C:\\Users\\Dell User\\Desktop\\DeepLearm\\Deep_Learning'

In [2]:
os.chdir(r"C:\Users\Dell User\Desktop\DeepLearm\Deep_Learning")

In [3]:
import tensorflow as tf

In [4]:
model = tf.keras.models.load_model("artifacts/training/training_model.h5")

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int
    params_is_augmentation: bool 

In [7]:
from src.constants import *
from src.utils.common import read_yaml,create_directories
import tensorflow as tf 
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/training_model.h5",
            training_data="artifacts/data_ingestion/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone",
            mlflow_uri="https://dagshub.com/gulmohargautam2005/Deep_Learning.mlflow",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE,
            params_is_augmentation=self.params.AUGMENTATION  
        )

        return eval_config

In [8]:
import os 
from src.utils.common import save_json
import tensorflow as tf 
from zipfile import ZipFile
import urllib.request as request
import mlflow
import mlflow.keras
from urllib.parse import urlparse

class ModelEvaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=4,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )
    @staticmethod
    def load_model(path:Path)->tf.keras.Model:
        return tf.keras.models.load_model(path)


    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = self.model.evaluate(self.valid_generator)
        self.save_score()


    def save_score(self):
        scores = {
            "loss": self.score[0],
            "accuracy": self.score[1]
        }

        save_json(
            path=Path("scores.json"),
            data=scores
        )
          

    
    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics(
                {"loss": self.score[0], "accuracy": self.score[1]}
            )
            # Model registry does not work with file store
            if tracking_url_type_store != "file":

                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                mlflow.keras.log_model(self.model, "model", registered_model_name="DenseNetModel")
            else:
                mlflow.keras.log_model(self.model, "model")


c:\Users\Dell User\Desktop\DeepLearm\Deep_Learning\.venv\lib\site-packages\mlflow\utils\requirements_utils.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [9]:
import os
for root, dirs, files in os.walk("."):
    for f in files:
        if "mlflow" in f.lower():
            print(os.path.join(root, f))

.\.venv\Lib\site-packages\mlflow\entities\_mlflow_object.py
.\.venv\Lib\site-packages\mlflow\entities\__pycache__\_mlflow_object.cpython-310.pyc
.\.venv\Lib\site-packages\mlflow\protos\mlflow_artifacts_pb2.py
.\.venv\Lib\site-packages\mlflow\store\artifact\mlflow_artifacts_repo.py
.\.venv\Lib\site-packages\mlflow\store\artifact\__pycache__\mlflow_artifacts_repo.cpython-310.pyc
.\.venv\Lib\site-packages\mlflow\utils\mlflow_tags.py
.\.venv\Lib\site-packages\mlflow\utils\__pycache__\mlflow_tags.cpython-310.pyc
.\.venv\Scripts\mlflow.exe


In [13]:
import sys
print(sys.executable)

c:\Users\Dell User\Desktop\DeepLearm\Deep_Learning\.venv\Scripts\python.exe


In [10]:
try:
    config=ConfigurationManager()
    eval_config=config.get_evaluation_config()
    evaluation=ModelEvaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()
except Exception as e:
    raise e





[2026-06-28 11:10:15,019: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-28 11:10:15,023: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-28 11:10:15,025: INFO: common: created directory at: artifacts]
Found 3732 images belonging to 4 classes.
Found 8714 images belonging to 4 classes.
933/933 [==============================] - 50s 45ms/step - loss: 0.5813 - accuracy: 0.8248
[2026-06-28 11:11:10,441: INFO: common: json file saved at: scores.json]


2026/06/28 11:11:13 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


[2026-06-28 11:11:39,491: WARNING: save: Found untraced functions such as _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op while saving (showing 5 of 120). These functions will not be directly callable after loading.]
INFO:tensorflow:Assets written to: C:\Users\DELLUS~1\AppData\Local\Temp\tmpaawr_4gt\model\data\model\assets
[2026-06-28 11:11:47,397: INFO: builder_impl: Assets written to: C:\Users\DELLUS~1\AppData\Local\Temp\tmpaawr_4gt\model\data\model\assets]


2026/06/28 11:12:18 WARNING mlflow.utils.requirements_utils: The following packages were not found in the public PyPI package index as of 2023-02-28; if these packages are not present in the public PyPI index, you must install them manually before loading your model: {'backports-tarfile'}
c:\Users\Dell User\Desktop\DeepLearm\Deep_Learning\.venv\lib\site-packages\_distutils_hack\__init__.py:30: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(
2026/06/28 11:12:18 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'DenseNetModel'.
2026/06/28 11:15:04 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model ver